# Content-Based recommendations

Run `01_data` and finish `01b_plots` in `experiments` mode first. Wikipedia was
confirmed with Andrea by Francesco on 18 November 2026. Use a GPU for faster embeddings.

For validation, only train ratings determine candidate eligibility, user profiles,
seen-item exclusion, tag availability and popularity fallback. The user profile is
the unweighted mean of unit embeddings of films rated >=4. Cosine similarity ranks
unseen candidates; ties use movieId. Short lists are kept and reported.

Accepted plots use a multilingual Sentence Transformer. Long plots are split into
token windows, encoded separately, averaged by token count and normalized. This
avoids silently using only a plot's opening. Missing plots use genres and the top
20 tags by frequency (alphabetical ties). Tags must be strictly earlier than their
author's last fitting rating; unknown authors and boundary ties are excluded.
This follows the project's per-user time protocol, not a global time cutoff.

Films with neither plots nor metadata have zero vectors and are excluded from
semantic rankings. Users without a usable liked-film profile receive train-only
Popularity recommendations; this fallback is recorded per user. Its score is a
rating count, while semantic scores are cosine similarities.

Static genres and Wikipedia revisions are current metadata, not historical
snapshots. Their provenance and languages are retained in `plots.parquet`.

In [ ]:
import importlib
import os
import subprocess
import sys
import types

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    from IPython import get_ipython

    shell = get_ipython()
    BRANCH = "feat/content-based"
    REPO_URL = "https://github.com/Cxsee/AI-in-Industry-project.git"
    EXTRAS = "content"
    DRIVE_DIR = "/content/drive/MyDrive/movierec"
    REPO_DIR = "/content/AI-in-Industry-project-content"

    drive.mount("/content/drive")
    os.environ["MOVIEREC_DATA_DIR"] = f"{DRIVE_DIR}/data"
    os.environ["MOVIEREC_RESULTS_DIR"] = f"{DRIVE_DIR}/results"
    os.makedirs(os.environ["MOVIEREC_DATA_DIR"], exist_ok=True)
    os.makedirs(os.environ["MOVIEREC_RESULTS_DIR"], exist_ok=True)

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)

    shell.run_line_magic("cd", REPO_DIR)
    subprocess.run(["git", "fetch", "-q", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    if not os.path.isfile(f"{REPO_DIR}/src/movierec/content.py"):
        raise RuntimeError(f"Branch {BRANCH!r} has no content module. Check BRANCH above.")
    target = f".[{EXTRAS}]" if EXTRAS else "."
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", target], check=True)

    # The running kernel doesn't pick up an editable install made after it started.
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")

    # Colab's IPython 7.34 still imports `imp`, which Python 3.13 removed.
    sys.modules.setdefault("imp", types.SimpleNamespace(reload=importlib.reload))
    shell.run_line_magic("load_ext", "autoreload")
    shell.run_line_magic("autoreload", "2")

In [ ]:
import json
import time
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import polars as pl
from IPython.display import display
from sentence_transformers import SentenceTransformer

from movierec import paths
from movierec.content import (
    encode_movie_texts,
    fitting_history,
    movie_texts,
    recommend_content,
)
from movierec.evaluation import evaluate_validation

EVALUATION_SPLIT = "val"
TOP_K = 10
MIN_RATINGS = 10
RELEVANT_RATING = 4.0
MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_REVISION = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
BATCH_SIZE = 32
MAX_TAGS = 20
DATA_DIR = paths.data_dir()
RESULTS_DIR = paths.results_dir()
PARAMS_PATH = RESULTS_DIR / "content_val_params.json"
PLOTS_PATH = DATA_DIR / "plots.parquet"

## Fitting data and metadata coverage

The full extraction must contain one row per `experiment_movies.parquet` ID,
including failed terminal extractions. Pending/retryable rows are not a finished
dataset. History films outside that catalog can still contribute to profiles via
genre/tag fallback. Candidate eligibility for validation is recomputed from train.

In [ ]:
required = [
    DATA_DIR / f"{name}.parquet"
    for name in ("ratings", "movies", "tags", "users_sample", "experiment_movies")
]
missing = [str(path) for path in [*required, PLOTS_PATH] if not path.is_file()]
if missing:
    raise FileNotFoundError("Finish 01_data and 01b_plots first. Missing: " + ", ".join(missing))

config = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "k": TOP_K,
    "min_ratings": MIN_RATINGS,
    "relevant_rating": RELEVANT_RATING,
    "max_tags": MAX_TAGS,
    "profile": "unweighted_mean_liked",
    "tag_cutoff": "strictly_before_author_last_fitting_rating",
    "chunk_pooling": "token_weighted_mean",
}
if EVALUATION_SPLIT == "test":
    saved = json.loads(PARAMS_PATH.read_text(encoding="utf-8"))["config"]
    if saved != config:
        raise ValueError("Test configuration differs from the one saved on validation.")

ratings = pl.scan_parquet(DATA_DIR / "ratings.parquet")
users = pl.read_parquet(DATA_DIR / "users_sample.parquet").sort("sample_rank")
history = fitting_history(ratings, EVALUATION_SPLIT)
catalog = (
    history.group_by("movieId")
    .len(name="n_ratings")
    .filter(pl.col("n_ratings") >= MIN_RATINGS)
    .collect()
)
sample_history = history.join(users.select("userId").lazy(), on="userId", how="semi").collect()
movies = pl.read_parquet(DATA_DIR / "movies.parquet")
plots = pl.read_parquet(PLOTS_PATH)
experiment_ids = pl.read_parquet(DATA_DIR / "experiment_movies.parquet").select("movieId")
if (
    plots["movieId"].n_unique() != plots.height
    or experiment_ids.join(plots, on="movieId", how="anti").height
    or plots.filter(
        pl.col("status").is_null() | pl.col("status").is_in(["pending", "retryable_error"])
    ).height
):
    raise ValueError(
        "plots.parquet must be a complete, unique extraction of the experiment catalog."
    )
liked_history_ids = sample_history.filter(pl.col("rating") >= RELEVANT_RATING).select("movieId")
needed = pl.concat([catalog.select("movieId"), liked_history_ids]).unique()
texts = movie_texts(
    movies.join(needed, on="movieId", how="semi"),
    plots,
    pl.scan_parquet(DATA_DIR / "tags.parquet"),
    history,
    max_tags=MAX_TAGS,
)
if needed.join(texts, on="movieId", how="anti").height:
    raise ValueError("A candidate/history film is missing from movies.parquet.")
display(texts.group_by("text_source").len())
display(texts.join(catalog, on="movieId", how="semi").group_by("text_source").len())
display(texts.filter(pl.col("text_source") == "plot").group_by("plot_language").len())

## Embeddings and recommendations

The model revision is pinned. Cache shards are identified by model, revision,
token-window rules, movie IDs and exact input texts, so changing the fitting
window or metadata invalidates stale embeddings. Completed shards survive restarts.
Model documentation: [multilingual MiniLM](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2),
[Sentence Transformer encoding](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html).

In [ ]:
start = time.perf_counter()
model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION)
embeddings = encode_movie_texts(
    texts,
    model,
    DATA_DIR / "content_embeddings",
    model_id=MODEL_ID,
    revision=MODEL_REVISION,
    batch_size=BATCH_SIZE,
    progress=lambda done, total: print(f"Embedding films: {done:,}/{total:,}", flush=True),
)
rankings, user_diagnostics = recommend_content(
    sample_history,
    users,
    texts["movieId"].to_numpy(),
    embeddings,
    catalog,
    k=TOP_K,
    relevant_rating=RELEVANT_RATING,
)
elapsed_seconds = time.perf_counter() - start
display(
    user_diagnostics.select(
        pl.len().alias("n_users"),
        pl.col("popularity_fallback").sum(),
        (pl.col("n_recommended") < TOP_K).sum().alias("n_short_rankings"),
        (pl.col("n_recommended") == 0).sum().alias("n_empty_rankings"),
    )
)
print(f"Recommendations: {rankings.height:,}; runtime including model load: {elapsed_seconds:.1f}s")

outputs = {
    f"content_{EVALUATION_SPLIT}.parquet": rankings,
    f"content_{EVALUATION_SPLIT}_user_diagnostics.parquet": user_diagnostics,
    f"content_{EVALUATION_SPLIT}_text_sources.parquet": texts.select(
        "movieId", "text_source", "plot_language"
    ),
}
if EVALUATION_SPLIT == "val":
    per_user, summary = evaluate_validation(
        ratings,
        users,
        movies,
        rankings,
        k=TOP_K,
        min_ratings=MIN_RATINGS,
        relevant_rating=RELEVANT_RATING,
    )
    outputs["content_val_user_metrics.parquet"] = per_user
    outputs["content_val_summary.parquet"] = summary.with_columns(
        pl.lit("content").alias("model"), pl.lit("val").alias("split")
    )
    display(summary)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name, frame in outputs.items():
    with TemporaryDirectory(dir=RESULTS_DIR) as folder:
        temporary = Path(folder) / name
        frame.write_parquet(temporary)
        temporary.replace(RESULTS_DIR / name)
    print(f"Saved {name}")
if EVALUATION_SPLIT == "val":
    manifest = {
        "config": config,
        "elapsed_seconds_including_cache": elapsed_seconds,
        "n_users": users.height,
        "catalog_size": catalog.height,
        "plot_file": PLOTS_PATH.name,
        "text_sources": texts.group_by("text_source").len().to_dicts(),
        "candidate_text_sources": texts.join(catalog, on="movieId", how="semi")
        .group_by("text_source")
        .len()
        .to_dicts(),
        "n_popularity_fallback_users": user_diagnostics["popularity_fallback"].sum(),
        "encoder_versions": {
            name: version(name) for name in ("sentence-transformers", "transformers", "torch")
        },
        "numpy_version": np.__version__,
        "polars_version": pl.__version__,
    }
    with TemporaryDirectory(dir=RESULTS_DIR) as folder:
        temporary = Path(folder) / PARAMS_PATH.name
        temporary.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        temporary.replace(PARAMS_PATH)

## Compare the three models

Run `07_comparison.ipynb` after this notebook. It includes `content_val.parquet`
when available and evaluates all models with the same users and protocol.
Do not choose hyperparameters on test. A test run requires the configuration saved
by a successful validation run and refits profiles and tag cutoffs on train + val.